# Feature Engineering, Normalization & Scaling Pipeline

This notebook implements feature preprocessing, cyclical encoding, normalization, and scaling on the cleaned Instacart dataset.
The workflow consists of:
1. **Feature Engineering**: Deriving cyclical trigonometric features (sine and cosine encodings for day and hour), basket position ratios, and purchase frequency indicators.
2. **Distribution & Scale Audit**: Assessing raw feature variances, ranges, and asymmetries that necessitate scaling.
3. **Comparative Scaling Methods**:
   - **StandardScaler**: Z-score standardization ($Z = \frac{x - \mu}{\sigma}$) to zero-center features with unit variance.
   - **MinMaxScaler**: Min-max normalization ($X_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$) to bound values within $[0, 1]$.
   - **RobustScaler**: Quantile-based scaling utilizing the median and interquartile range ($X_{\text{robust}} = \frac{x - Q_2}{\text{IQR}}$) for outlier robustness.
4. **Leakage-Free Train/Test Split**: Enforcing a strict 80/20 stratified split where scalers are fitted exclusively on the training partition.
5. **Empirical Model Validation**: Training a regularized Logistic Regression classifier to evaluate convergence and discriminative power ($ROC\text{-}AUC$).
6. **Processed Data Export**: Persisting the transformed feature dataset to `data/processed/scaled_features.csv`.

In [1]:
pd=__import__('pandas')
np=__import__('numpy')
StandardScaler=__import__('sklearn.preprocessing').preprocessing.StandardScaler
MinMaxScaler=__import__('sklearn.preprocessing').preprocessing.MinMaxScaler
RobustScaler=__import__('sklearn.preprocessing').preprocessing.RobustScaler
train_test_split=__import__('sklearn.model_selection').model_selection.train_test_split
LogisticRegression=__import__('sklearn.linear_model').linear_model.LogisticRegression
roc_auc_score=__import__('sklearn.metrics').metrics.roc_auc_score
print(pd.__name__,StandardScaler.__name__)

pandas StandardScaler


### 1. Ingest Cleaned Dataset
We load the validated cleaned dataset from `data/processed/cleaned_instacart_data.csv`.

In [2]:
df=pd.read_csv('data/processed/cleaned_instacart_data.csv')
print(df.shape)
df.head(3)

(100000, 17)


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,department,aisle,is_first_order,add_to_cart_order_raw
0,1,49302,1.0,1,112108,train,4,4,10,9.0,bulgarian yogurt,120,16,dairy eggs,yogurt,0,1
1,1,11109,2.0,1,112108,train,4,4,10,9.0,organic 4% milk fat whole milk cottage cheese,108,16,dairy eggs,other creams cheeses,0,2
2,1,10246,3.0,0,112108,train,4,4,10,9.0,organic celery hearts,83,4,produce,fresh vegetables,0,3


### 2. Feature Engineering: Cyclical Encodings & Relative Cart Position
Linear and neural models struggle with raw cyclical variables like day-of-week (0 to 6) and hour-of-day (0 to 23) because day 6 (Saturday) is numerically far from day 0 (Sunday) despite being temporally adjacent.
We map these features onto continuous circular coordinates:
$$\text{dow}_{\sin} = \sin\left(\frac{2\pi \cdot \text{dow}}{7}\right), \quad \text{dow}_{\cos} = \cos\left(\frac{2\pi \cdot \text{dow}}{7}\right)$$
$$\text{hour}_{\sin} = \sin\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad \text{hour}_{\cos} = \cos\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$
Additionally, we compute `cart_ratio` as $\frac{\text{add\_to\_cart\_order}}{\text{basket\_size}}$, normalizing sequence across baskets of varying dimensions.

In [3]:
df['order_dow_sin']=np.sin(2*np.pi*df['order_dow']/7.0)
df['order_dow_cos']=np.cos(2*np.pi*df['order_dow']/7.0)
df['order_hour_sin']=np.sin(2*np.pi*df['order_hour_of_day']/24.0)
df['order_hour_cos']=np.cos(2*np.pi*df['order_hour_of_day']/24.0)
basket_sizes=df.groupby('order_id')['product_id'].transform('count')
df['cart_ratio']=df['add_to_cart_order']/basket_sizes
print(df[['order_dow_sin','order_dow_cos','order_hour_sin','order_hour_cos','cart_ratio']].head())

   order_dow_sin  order_dow_cos  order_hour_sin  order_hour_cos  cart_ratio
0      -0.433884      -0.900969             0.5       -0.866025       0.125
1      -0.433884      -0.900969             0.5       -0.866025       0.250
2      -0.433884      -0.900969             0.5       -0.866025       0.375
3      -0.433884      -0.900969             0.5       -0.866025       0.500
4      -0.433884      -0.900969             0.5       -0.866025       0.625


### 3. Feature Scale Inspection Prior to Scaling
We examine the summary statistics across all continuous numeric predictors to assess scale disparities.
Features like `order_number` range from 1 to 100, while `days_since_prior_order` ranges from 0 to 30, and `cart_ratio` ranges from 0.01 to 1.0.
Without normalization, gradient descent steps are dominated by high-variance features, skewing regularization penalties ($\ell_1$ and $\ell_2$) and impairing distance-based metrics.

In [4]:
feature_cols=['order_number','days_since_prior_order','add_to_cart_order','is_first_order','order_dow_sin','order_dow_cos','order_hour_sin','order_hour_cos','cart_ratio']
stats_before=df[feature_cols].describe().T[['mean','std','min','max']]
print(stats_before)

                             mean        std       min         max
order_number            17.289540  16.685289  4.000000  100.000000
days_since_prior_order  17.083840  10.440503  0.000000   30.000000
add_to_cart_order        8.522430   6.620438  1.000000   25.500000
is_first_order           0.000000   0.000000  0.000000    0.000000
order_dow_sin           -0.006464   0.677293 -0.974928    0.974928
order_dow_cos            0.163349   0.717327 -0.900969    1.000000
order_hour_sin          -0.230556   0.642825 -1.000000    1.000000
order_hour_cos          -0.491750   0.540193 -1.000000    1.000000
cart_ratio               0.541541   0.285745  0.014286    1.000000


### 4. Leakage-Free Stratified Train/Test Partitioning
To guarantee rigorous model evaluation and prevent data leakage, we partition the dataset into 80% training and 20% test subsets before fitting any scaler.
Stratification on `reordered` ensures identical positive class proportions in both partitions.

In [5]:
X=df[feature_cols]
y=df['reordered']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)
print(y_train.mean(),y_test.mean())

(80000, 9) (20000, 9)
0.600275 0.60025


### 5. Implementation of Scaling Techniques
We fit three standard scalers on the training set:
1. **StandardScaler**: Centers each feature to mean 0 and standard deviation 1.
2. **MinMaxScaler**: Compresses features into a rigid $[0, 1]$ interval.
3. **RobustScaler**: Scales using the median and Interquartile Range ($IQR$), safeguarding against residual extreme values.
Scalers are fitted strictly on `X_train` and subsequently applied to `X_test` via `.transform()`.

In [6]:
std_scaler=StandardScaler()
minmax_scaler=MinMaxScaler()
robust_scaler=RobustScaler()
X_train_std=std_scaler.fit_transform(X_train)
X_test_std=std_scaler.transform(X_test)
X_train_minmax=minmax_scaler.fit_transform(X_train)
X_test_minmax=minmax_scaler.transform(X_test)
X_train_robust=robust_scaler.fit_transform(X_train)
X_test_robust=robust_scaler.transform(X_test)
df_scaled_summary=pd.DataFrame({'std_mean':X_train_std.mean(axis=0),'std_variance':X_train_std.var(axis=0),'minmax_min':X_train_minmax.min(axis=0),'minmax_max':X_train_minmax.max(axis=0)},index=feature_cols)
print(df_scaled_summary)

                            std_mean  std_variance  minmax_min  minmax_max
order_number           -3.090861e-17           1.0         0.0         1.0
days_since_prior_order  3.970158e-17           1.0         0.0         1.0
add_to_cart_order      -2.913225e-17           1.0         0.0         1.0
is_first_order          0.000000e+00           0.0         0.0         0.0
order_dow_sin          -5.098144e-17           1.0         0.0         1.0
order_dow_cos          -4.618528e-17           1.0         0.0         1.0
order_hour_sin         -2.202682e-17           1.0         0.0         1.0
order_hour_cos          1.145750e-16           1.0         0.0         1.0
cart_ratio              5.524470e-17           1.0         0.0         1.0


### 6. Empirical Validation: Baseline Classification on Scaled Features
We train an $\ell_2$-regularized Logistic Regression model on both unscaled and standardized features to evaluate parameter convergence and classification performance.
We record ROC-AUC, demonstrating effective discriminative capability on the binary `reordered` task.

In [7]:
model_unscaled=LogisticRegression(max_iter=1000,random_state=42)
model_unscaled.fit(X_train,y_train)
auc_unscaled=roc_auc_score(y_test,model_unscaled.predict_proba(X_test)[:,1])
model_std=LogisticRegression(max_iter=1000,random_state=42)
model_std.fit(X_train_std,y_train)
auc_std=roc_auc_score(y_test,model_std.predict_proba(X_test_std)[:,1])
print(auc_unscaled,auc_std)

0.6960793175868195 0.6960680027266105


### 7. Export Scaled Feature Artifact
The standardized feature matrix and corresponding target variable are consolidated and written to `data/processed/scaled_features.csv` for modeling and downstream experiments.

In [8]:
scaled_export=pd.DataFrame(X_train_std,columns=feature_cols)
scaled_export['reordered']=y_train.values
scaled_export.to_csv('data/processed/scaled_features.csv',index=False)
print(scaled_export.shape)
print(scaled_export.head(3))

(80000, 10)
   order_number  days_since_prior_order  add_to_cart_order  is_first_order  \
0      0.704261               -1.350672           2.184433             0.0   
1     -0.497406               -1.350672          -0.985852             0.0   
2     -0.196989                0.661463          -0.080056             0.0   

   order_dow_sin  order_dow_cos  order_hour_sin  order_hour_cos  cart_ratio  \
0       0.007694       1.167767       -0.044274       -0.878735    1.461052   
1      -1.145940       0.642729       -1.143025        0.434089   -1.394605   
2      -0.632525      -1.483109        0.760067       -0.878735   -0.027224   

   reordered  
0          1  
1          1  
2          1  
